# S4b - SQL exercise

I repeat (again) :
- always build your queries step by step : start small, and make them more and more elaborate
- read the course
- first formulate the operations you want to perform in order, then look for how to implement them in SQL (course, tutorial, doc, stackoverflow, etc.)

You will find 5 `.csv` files in the folder `data/exercise-hospitals` 

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/DM-SQL/DM-SQL.sqlite')
c = conn.cursor()


In [2]:
def add_table(connector: object, path: 'string', table_name: 'string'):
    df = pd.read_csv(path)
    df.to_sql(table_name, connector, if_exists='replace', index=False)

## 1. Load them in a database by a method of your choice :

In [3]:
import os

csv_path = os.path.join('.','data','DM-SQL')
file_names = [f for f in os.listdir(csv_path) if f.endswith('.csv')]
for f in file_names:
    add_table(conn, csv_path + '/' + f, f[:-4])

In [4]:
def exe(cursor: object, query: 'string'):
    cursor.execute(query)
    for row in  cursor.fetchall():
        print(row)

## 2. List the tables and columns in each table. Create an ERD. Indicate the primary key and any foreign keys for each table. Write down the different relationships and their types (one-to-one, one-to-many, etc.)

In [5]:
list_table = '''
PRAGMA table_list;
'''

exe(c, list_table)

('main', 'units', 'table', 4, 0, 0)
('main', 'hospitals', 'table', 3, 0, 0)
('main', 'payment', 'table', 4, 0, 0)
('main', 'admissions', 'table', 5, 0, 0)
('main', 'patients', 'table', 4, 0, 0)
('main', 'sqlite_schema', 'table', 5, 0, 0)
('temp', 'sqlite_temp_schema', 'table', 5, 0, 0)


In [6]:
# list all columns of all tables

# first, get table name
c.execute(list_table)
for row in c.fetchall():
    table = row[1] # to improve readability

    # then get columns names
    column_list = 'PRAGMA table_info(' + table +')'
    print('\n----- table ' + row[1] + ' columns -----\n')
    c.execute(column_list)
    
    # print columns names
    for row in  c.fetchall():
        column = row[1] # to improve readability
        print(column)
    
    # finally print first lines of each table   
    print('\n table ' + table + ' first lines :\n')
    select_all = 'SELECT * FROM ' + table + ' LIMIT 5'
    exe(c, select_all)


----- table units columns -----

id
names
id_hospital
beds

 table units first lines :

(1, 'Gastro-entérologie', 1, 10)
(2, 'ORL', 1, 49)
(3, 'Cardiologie', 1, 11)
(4, 'Psychiatrie', 1, 14)
(5, 'Néphrologie', 2, 10)

----- table hospitals columns -----

id
name
adresse

 table hospitals first lines :

(1, 'Centre Hospitalier Universitaire de Bordeaux', 'Place Amélie Raba Léon, 33000 Bordeaux')
(2, 'Hôpital Saint-Louis', '1 Avenue Claude Vellefaux, 75010 Paris')
(3, 'Centre Hospitalier Universitaire de Strasbourg', "1 Place de l'Hôpital, 67091 Strasbourg")
(4, 'Centre Hospitalier Universitaire de Montpellier', '191 Avenue du Doyen Gaston Giraud, 34295 Montpellier')
(5, 'Hôpital Necker-Enfants Malades', '149 Rue de Sèvres, 75015 Paris')

----- table payment columns -----

id
admission_id
amount
method

 table payment first lines :

(1, 1, '1905.61€', 'check')
(2, 2, '4909.56€', 'check')
(3, 3, '1586.04€', 'pending')
(4, 4, '4696.34€', 'pending')
(5, 5, '2333.6€', 'cash')

----- table a

## 3. How many hospitals is there in the database ?

In [7]:
n_hospitals = 'SELECT COUNT(*) FROM hospitals'

exe(c, n_hospitals)

(20,)


## 4.How many hospital units of each type are there, overall (all hospitals combined)?

ex. output:
```
('Cardiologie', 17)
('Gastro-entérologie', 12)
etc.
```

In [10]:
n_different_units = 'SELECT names, COUNT(*) FROM units GROUP BY names'

exe(c, n_different_units)

('Cardiologie', 17)
('Gastro-entérologie', 12)
('Néphrologie', 17)
('ORL', 15)
('Oncologie', 12)
('Psychiatrie', 17)


## 5. List the different units by hospitals (hospitals names and units names should both appear in the list) :

ex. output :

```
('Centre Hospitalier Universitaire de Bordeaux', 'Cardiologie')
('Centre Hospitalier Universitaire de Bordeaux', 'Gastro-entérologie')
('Centre Hospitalier Universitaire de Bordeaux', 'ORL')
('Centre Hospitalier Universitaire de Bordeaux', 'Psychiatrie')
('Hôpital Saint-Louis', 'Cardiologie')
('Hôpital Saint-Louis', 'Gastro-entérologie')
etc.
``` 

In [13]:
units_list = '''
SELECT 
    h.name, u.names 
FROM hospitals AS h
JOIN
    units as u
    ON
        h.id = u.id_hospital
'''

exe(c, units_list)

('Centre Hospitalier Universitaire de Bordeaux', 'Cardiologie')
('Centre Hospitalier Universitaire de Bordeaux', 'Gastro-entérologie')
('Centre Hospitalier Universitaire de Bordeaux', 'ORL')
('Centre Hospitalier Universitaire de Bordeaux', 'Psychiatrie')
('Hôpital Saint-Louis', 'Cardiologie')
('Hôpital Saint-Louis', 'Gastro-entérologie')
('Hôpital Saint-Louis', 'Néphrologie')
('Hôpital Saint-Louis', 'ORL')
('Hôpital Saint-Louis', 'Oncologie')
('Hôpital Saint-Louis', 'Psychiatrie')
('Centre Hospitalier Universitaire de Strasbourg', 'Cardiologie')
('Centre Hospitalier Universitaire de Strasbourg', 'Gastro-entérologie')
('Centre Hospitalier Universitaire de Strasbourg', 'Néphrologie')
('Centre Hospitalier Universitaire de Strasbourg', 'ORL')
('Centre Hospitalier Universitaire de Strasbourg', 'Oncologie')
('Centre Hospitalier Universitaire de Strasbourg', 'Psychiatrie')
('Centre Hospitalier Universitaire de Montpellier', 'Cardiologie')
('Centre Hospitalier Universitaire de Montpellier', 'O

## 6. How many units in each hospital :

ex. output :

```
('Centre Hospitalier Universitaire de Bordeaux', 4)
('Centre Hospitalier Universitaire de Caen', 5)
('Centre Hospitalier Universitaire de Clermont-Ferrand', 3)
etc.
```

In [15]:
n_hospitals_units = '''
SELECT 
    h.name, COUNT(u.names) 
FROM hospitals AS h
JOIN
    units as u
    ON
        h.id = u.id_hospital
GROUP BY
    h.name
'''

exe(c, n_hospitals_units)

('Centre Hospitalier Universitaire de Bordeaux', 4)
('Centre Hospitalier Universitaire de Caen', 5)
('Centre Hospitalier Universitaire de Clermont-Ferrand', 3)
('Centre Hospitalier Universitaire de Dijon', 4)
('Centre Hospitalier Universitaire de Grenoble', 6)
('Centre Hospitalier Universitaire de Lille', 6)
('Centre Hospitalier Universitaire de Limoges', 6)
('Centre Hospitalier Universitaire de Lyon', 3)
('Centre Hospitalier Universitaire de Montpellier', 3)
('Centre Hospitalier Universitaire de Nantes', 4)
('Centre Hospitalier Universitaire de Nice', 3)
('Centre Hospitalier Universitaire de Rennes', 6)
('Centre Hospitalier Universitaire de Rouen', 5)
('Centre Hospitalier Universitaire de Strasbourg', 6)
('Centre Hospitalier Universitaire de Toulouse', 3)
('Hôpital Bichat-Claude Bernard', 6)
('Hôpital Necker-Enfants Malades', 4)
('Hôpital Saint-Louis', 6)
('Hôpital de la Pitié-Salpêtrière', 4)
('Hôpital de la Timone', 3)


## 7. There are two ways to answer this question: a complex one that yields a bonus, and a longer but easier one (so no bonus). Choose between A or B:

**A.** Write a query that associates each hospital with its department of origin (number):

* You need to extract the first two digits of the postal code (address column)
* Hint 1: To determine the position of the postal code in the address, observe the text and see if a character, always placed in the same location, would allow you to identify the position of the postal code
* Hint 2: The postal code is always 5 characters long, and we're interested in the first two
* Hint 3: You may need to use the same function twice to indicate where the substring begins and ends

Recommendation: When writing a complex query, work your way through it.
* Start by writing a query that lists the hospital addresses
* Then add instructions to this query that remove the beginning of the text up to the postal code
* Then add instructions to display only the portion of text (numbers) you want
* Etc.

**B.** If you can't answer this question the first way (A, which is difficult), add a "department" column by hand to the hospital table (using the appropriate queries of course)

ex. output :

```
('Centre Hospitalier Universitaire de Bordeaux', '33')
('Hôpital Saint-Louis', '75')
('Centre Hospitalier Universitaire de Strasbourg', '67')
('Centre Hospitalier Universitaire de Montpellier', '34')
etc.
```

In [24]:
n_hospitals_depcodes = '''
SELECT 
    name, SUBSTR(adresse,
        INSTR(adresse, ',') + 2,
        LENGTH(INSTR(adresse, ',') + 4))
FROM hospitals 
'''

exe(c, n_hospitals_depcodes)

('Centre Hospitalier Universitaire de Bordeaux', '33')
('Hôpital Saint-Louis', '75')
('Centre Hospitalier Universitaire de Strasbourg', '67')
('Centre Hospitalier Universitaire de Montpellier', '34')
('Hôpital Necker-Enfants Malades', '75')
('Centre Hospitalier Universitaire de Toulouse', '31')
('Hôpital de la Timone', '13')
('Centre Hospitalier Universitaire de Nice', '06')
('Centre Hospitalier Universitaire de Nantes', '44')
('Hôpital de la Pitié-Salpêtrière', '75')
('Centre Hospitalier Universitaire de Rennes', '35')
('Centre Hospitalier Universitaire de Lille', '59')
('Centre Hospitalier Universitaire de Grenoble', '38')
('Hôpital Bichat-Claude Bernard', '75')
('Centre Hospitalier Universitaire de Clermont-Ferrand', '63')
('Centre Hospitalier Universitaire de Dijon', '21')
('Centre Hospitalier Universitaire de Lyon', '69')
('Centre Hospitalier Universitaire de Rouen', '76')
('Centre Hospitalier Universitaire de Caen', '14')
('Centre Hospitalier Universitaire de Limoges', '87')


## 8. Are there multiple hospitals in the same departments? List the number of hospitals per department, listed in ascending order by department number:

ex. output :

```
('06', 1)
('13', 1)
('14', 1)
('21', 1)
etc.
```

In [27]:
n_hospitals_depcodes = '''
SELECT 
    SUBSTR(adresse,
        INSTR(adresse, ',') + 2,
        LENGTH(INSTR(adresse, ',') + 4)) AS department,
    COUNT(name)
    
FROM hospitals 

GROUP BY department
ORDER BY department ASC

'''

exe(c, n_hospitals_depcodes)

('06', 1)
('13', 1)
('14', 1)
('21', 1)
('31', 1)
('33', 1)
('34', 1)
('35', 1)
('38', 1)
('44', 1)
('59', 1)
('63', 1)
('67', 1)
('69', 1)
('75', 4)
('76', 1)
('87', 1)


## 9. List the number of hospital services in each department, and only display those with more than 4 (order the results by decreasing number of services):

ex. output:

```
('75', 20)
('87', 6)
('67', 6)
etc.
```

In [33]:
n_services_dep = '''
SELECT 
    SUBSTR(h.adresse,
        INSTR(h.adresse, ',') + 2,
        LENGTH(INSTR(h.adresse, ',') + 4)) AS department,
        COUNT(u.names) as n
FROM hospitals AS h
JOIN
    units as u
    ON
        h.id = u.id_hospital
GROUP BY
    department
HAVING
    n > 4
ORDER BY 
    n DESC
'''

exe(c, n_services_dep)

('75', 20)
('87', 6)
('67', 6)
('59', 6)
('38', 6)
('35', 6)
('76', 5)
('14', 5)


## 10. Admissions have a date of entry into the hospital, and a date of discharge / release. Calculate the average length (duration) of all admissions recorded in the database (in days, rounded to one decimal place).

In [39]:
duration_mean = '''
SELECT 
    ROUND(AVG(CAST((JULIANDAY(release_date) - JULIANDAY(entry_date)) AS INTEGER)),1) AS mean_duration

FROM 
    admissions
'''

exe(c, duration_mean)

(14.6,)


## 11. Calculate the average length of a cardiology admission (all hospitals combined).

In [44]:
duration_mean_cardio = '''
SELECT 
    u.names,
    ROUND(AVG(CAST((JULIANDAY(a.release_date) - JULIANDAY(a.entry_date)) AS INTEGER)),1) AS mean_duration
FROM 
    admissions AS a
JOIN
    units AS u
    ON
        a.unit_id = u.id
GROUP BY u.names
HAVING u.names = 'Cardiologie'
'''

exe(c, duration_mean_cardio)

('Cardiologie', 14.2)


## 12. Compare (list) the average length of time for a cardiology admission between all hospitals with a cardiology department. Rank the results in descending order of this average length.

ex. output:

```
('Hôpital de la Timone', 'Cardiologie', 16.5)
('Centre Hospitalier Universitaire de Caen', 'Cardiologie', 16.0)
('Centre Hospitalier Universitaire de Bordeaux', 'Cardiologie', 15.2)
('Centre Hospitalier Universitaire de Strasbourg', 'Cardiologie', 15.2)
etc.
```

In [47]:
comp_duration_mean_cardio = '''
SELECT 
    h.name,
    u.names,
    ROUND(AVG(CAST((JULIANDAY(a.release_date) - JULIANDAY(a.entry_date)) AS INTEGER)),1) AS mean_duration
FROM 
    admissions AS a
JOIN
    units AS u
    ON
        a.unit_id = u.id
JOIN
    hospitals AS h
    ON
        u.id_hospital = h.id
GROUP BY 
    h.name,
    u.names
HAVING u.names = 'Cardiologie'
ORDER BY
    mean_duration DESC
'''

exe(c, comp_duration_mean_cardio)

('Hôpital de la Timone', 'Cardiologie', 16.5)
('Centre Hospitalier Universitaire de Caen', 'Cardiologie', 16.0)
('Centre Hospitalier Universitaire de Bordeaux', 'Cardiologie', 15.2)
('Centre Hospitalier Universitaire de Strasbourg', 'Cardiologie', 15.2)
('Hôpital Bichat-Claude Bernard', 'Cardiologie', 14.8)
('Centre Hospitalier Universitaire de Rennes', 'Cardiologie', 14.6)
('Hôpital Saint-Louis', 'Cardiologie', 14.6)
('Hôpital de la Pitié-Salpêtrière', 'Cardiologie', 14.5)
('Centre Hospitalier Universitaire de Lille', 'Cardiologie', 14.4)
('Centre Hospitalier Universitaire de Limoges', 'Cardiologie', 14.1)
('Centre Hospitalier Universitaire de Lyon', 'Cardiologie', 14.0)
('Centre Hospitalier Universitaire de Montpellier', 'Cardiologie', 13.6)
('Centre Hospitalier Universitaire de Rouen', 'Cardiologie', 13.2)
('Centre Hospitalier Universitaire de Toulouse', 'Cardiologie', 12.6)
('Centre Hospitalier Universitaire de Nantes', 'Cardiologie', 12.2)
('Centre Hospitalier Universitaire de Gre

## 13. What query displays the 25% of hospitals with the shortest hospital stays for cardiology? Present the output as follows:

```
('Cardiologie', 'Centre Hospitalier Universitaire de Clermont-Ferrand', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Grenoble', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Nantes', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Toulouse', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Rouen', '<25%')
etc.
```



In [56]:
quart = '''
SELECT
    names,
    name,
    Quartile

FROM
    (
    SELECT 
        names,
        name,
        CASE NTILE(4) OVER (
            ORDER BY mean_duration
            )
            WHEN 1
                THEN '<25%'
            WHEN 2
                THEN '2d'
            WHEN 3
                THEN '3rd'
            WHEN 4
                THEN '4th'
            END
                AS Quartile
    
    FROM (
        SELECT 
            h.name,
            u.names,
            ROUND(AVG(CAST((JULIANDAY(a.release_date) - JULIANDAY(a.entry_date)) AS INTEGER)),1) AS mean_duration
        FROM 
            admissions AS a
        JOIN
            units AS u
            ON
                a.unit_id = u.id
        JOIN
            hospitals AS h
            ON
                u.id_hospital = h.id
        GROUP BY 
            h.name,
            u.names
        HAVING u.names = 'Cardiologie'
        ORDER BY
            mean_duration DESC
)
)

WHERE
    Quartile LIKE '<25%'
'''

exe(c, quart)

('Cardiologie', 'Centre Hospitalier Universitaire de Clermont-Ferrand', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Grenoble', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Nantes', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Toulouse', '<25%')
('Cardiologie', 'Centre Hospitalier Universitaire de Rouen', '<25%')
